In [2]:
#import required libraries
import numpy as np

#use for reading our saved configuration
import json
import tensorflow as tf
from pathlib import Path

#Sequential lets us stack neural network layers
from tensorflow.keras.models import Sequential

#Layers required for our LSTM model
from tensorflow.keras.layers import (
    Input,
    Embedding,
    LSTM,
    Dense
)

print("Libraries imported successfully")

Libraries imported successfully


In [4]:
#Load training and validation Data
processed_dir = Path("../data/processed")

#Load training data
X_train = np.load(
    processed_dir / "X_train.npy"
)
#Each value is the correct next word
Y_train = np.load(
    processed_dir / "Y_train.npy"
)
#Load Validation Data
X_val = np.load(
    processed_dir / "X_val.npy"
)
Y_val = np.load(
    processed_dir / "Y_val.npy"
)
print("X_train:", X_train.shape)
print("Y_train:", Y_train.shape)

print("X_val:",X_val.shape)
print("Y_val:",Y_val.shape)

X_train: (2153888, 20)
Y_train: (2153888,)
X_val: (237549, 20)
Y_val: (237549,)


In [6]:
with open(
    processed_dir / "config.json",
    "r"
) as file:
    config = json.load(file)

#number of words the model can predict 
VOCAB_SIZE = config["vocab_size"]
#Number of previous words provided to the model
CONTEXT_LENGTH = config["context_length"]

print("Vocabulary size:", VOCAB_SIZE)
print("Context length:", CONTEXT_LENGTH)

Vocabulary size: 20000
Context length: 20


In [13]:
#Build the LSTM model
#clear old keras objects/models from money
tf.keras.backend.clear_session()

EMBEDDING_DIM = 128

LSTM_UNITS = 128

lstm_model = Sequential([
    #input shape:
    #(batch_size, 20)
    Input(shape=(CONTEXT_LENGTH,)),
    #Output: 
    #(batch_size, 20, 128)
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
    #LSTM reads all 20 words sequentially 
    LSTM(
        units = LSTM_UNITS,
        return_sequences=False
    ),
    #Output
    Dense(
        VOCAB_SIZE,
        activation="softmax"
    )

])
print("Lstm model created successfully")

Lstm model created successfully


In [9]:
#View model architecture
lstm_model.summary()
print("\n Final output shape:")
print(lstm_model.output_shape)

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 20, 128)        │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 20000)          │     2,580,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,271,584 (20.11 MB)

 Trainable params: 5,271,584 (20.11 MB)

 Non-trainable params: 0 (0.00 B)


 Final output shape:
(None, 20000)


In [15]:
#Compile LSTM
lstm_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        ),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(
            k=5,
            name="top_5_accuracy"
        )
    ],
    jit_compile=False
)
print("LSTM compiled succesfully")

LSTM compiled succesfully


In [12]:
#Small batch test
test_result = lstm_model.train_on_batch(
    X_train[:256],
    Y_train[:256]
)

print("One-batch test successful")
print("Loss:", test_result[0])
print("Accuracy:", test_result[1])
print("Top 5 accuracy:", test_result[2])

One-batch test successful
Loss: 9.903557777404785
Accuracy: 0.0
Top 5 accuracy: 0.0


In [16]:
#Create LSTM callbacks
from tensorflow.keras.callbacks import(
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)
#Folder where models are stored
model_dir = Path("../models")
model_dir.mkdir(exist_ok=True)

#Early Stopping
lstm_early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)
#Model checkpoint
lstm_checkpoint = ModelCheckpoint(
    model_dir / "best_lstm.keras",
    monitor="val_loss",
    save_best_only=True
)
#Reduce learning rate when validation improvement stalls
lstm_reduce_lr = ReduceLROnPlateau(
    monitor ="val_loss",
    factor = 0.5,
    patience=1,
    min_lr=1e-6
) 

print("LSTM callbacks created")

LSTM callbacks created


In [19]:
#Train LSTM on FULL DATASET
history_lst = lstm_model.fit(
    X_train,
    Y_train,
    batch_size=256,
    epochs=10,
    validation_data=(
        X_val,
        Y_val
    ),
    #LSTM specific callbacks
    callbacks=[
        lstm_early_stopping,
        lstm_checkpoint,
        lstm_reduce_lr

    ]
)

Epoch 1/10
8414/8414 ━━━━━━━━━━━━━━━━━━━━ 461s 55ms/step - accuracy: 0.1466 - loss: 6.1274 - top_5_accuracy: 0.2778 - val_accuracy: 0.1883 - val_loss: 5.5312 - val_top_5_accuracy: 0.3396 - learning_rate: 0.0010
Epoch 2/10
8414/8414 ━━━━━━━━━━━━━━━━━━━━ 462s 55ms/step - accuracy: 0.2052 - loss: 5.2393 - top_5_accuracy: 0.3643 - val_accuracy: 0.2126 - val_loss: 5.1541 - val_top_5_accuracy: 0.3789 - learning_rate: 0.0010
Epoch 3/10
8414/8414 ━━━━━━━━━━━━━━━━━━━━ 485s 58ms/step - accuracy: 0.2281 - loss: 4.8479 - top_5_accuracy: 0.4000 - val_accuracy: 0.2244 - val_loss: 4.9858 - val_top_5_accuracy: 0.3981 - learning_rate: 0.0010
Epoch 4/10
8414/8414 ━━━━━━━━━━━━━━━━━━━━ 491s 58ms/step - accuracy: 0.2442 - loss: 4.5971 - top_5_accuracy: 0.4251 - val_accuracy: 0.2317 - val_loss: 4.9034 - val_top_5_accuracy: 0.4097 - learning_rate: 0.0010
Epoch 5/10
8414/8414 ━━━━━━━━━━━━━━━━━━━━ 611s 73ms/step - accuracy: 0.2575 - loss: 4.4121 - top_5_accuracy: 0.4444 - val_accuracy: 0.2367 - val_loss: 4.863

In [22]:
#Save lstm history
import pandas as pd

results_dir = Path("../results")
results_dir.mkdir(exist_ok=True)

#Convert the training history into a table
lstm_history_df = pd.DataFrame(
    history_lstm.history
)
lstm_history_df.to_csv(
    results_dir / "lstm_training_history.csv",
    index=False
)

print(lstm_history_df)
print("\nLSTM training history saved")

NameError: name 'history_lstm' is not defined

In [23]:
# ============================================================
# SAVE / RECONSTRUCT LSTM TRAINING HISTORY
# ============================================================

import pandas as pd
from pathlib import Path


# These values are taken from the LSTM training output
# printed after each epoch.

lstm_history_data = {

    "accuracy": [
        0.1466,
        0.2052,
        0.2281,
        0.2442,
        0.2575,
        0.2694,
        0.2802,
        0.2909,
        0.3094
    ],

    "loss": [
        6.1274,
        5.2393,
        4.8479,
        4.5971,
        4.4121,
        4.2640,
        4.1399,
        4.0339,
        3.8554
    ],

    "top_5_accuracy": [
        0.2778,
        0.3643,
        0.4000,
        0.4251,
        0.4444,
        0.4612,
        0.4757,
        0.4892,
        0.5112
    ],

    "val_accuracy": [
        0.1883,
        0.2126,
        0.2244,
        0.2317,
        0.2367,
        0.2404,
        0.2438,
        0.2463,
        0.2497
    ],

    "val_loss": [
        5.5312,
        5.1541,
        4.9858,
        4.9034,
        4.8630,
        4.8421,
        4.8359,
        4.8433,
        4.8372
    ],

    "val_top_5_accuracy": [
        0.3396,
        0.3789,
        0.3981,
        0.4097,
        0.4164,
        0.4221,
        0.4255,
        0.4274,
        0.4307
    ],

    "learning_rate": [
        0.001,
        0.001,
        0.001,
        0.001,
        0.001,
        0.001,
        0.001,
        0.001,
        0.0005
    ]
}


# Convert the dictionary into a pandas table
lstm_history_df = pd.DataFrame(lstm_history_data)


# Add human-readable epoch numbers: 1 to 9
lstm_history_df.insert(
    0,
    "epoch",
    range(1, len(lstm_history_df) + 1)
)


# Create the results folder if needed
results_dir = Path("../results")
results_dir.mkdir(exist_ok=True)


# Save the table
lstm_history_df.to_csv(
    results_dir / "lstm_training_history.csv",
    index=False
)


print(lstm_history_df)

print("\nLSTM training history saved successfully.")

   epoch  accuracy    loss  top_5_accuracy  val_accuracy  val_loss  \
0      1    0.1466  6.1274          0.2778        0.1883    5.5312   
1      2    0.2052  5.2393          0.3643        0.2126    5.1541   
2      3    0.2281  4.8479          0.4000        0.2244    4.9858   
3      4    0.2442  4.5971          0.4251        0.2317    4.9034   
4      5    0.2575  4.4121          0.4444        0.2367    4.8630   
5      6    0.2694  4.2640          0.4612        0.2404    4.8421   
6      7    0.2802  4.1399          0.4757        0.2438    4.8359   
7      8    0.2909  4.0339          0.4892        0.2463    4.8433   
8      9    0.3094  3.8554          0.5112        0.2497    4.8372   

   val_top_5_accuracy  learning_rate  
0              0.3396         0.0010  
1              0.3789         0.0010  
2              0.3981         0.0010  
3              0.4097         0.0010  
4              0.4164         0.0010  
5              0.4221         0.0010  
6              0.4255     